TBA

### import packages

In [1]:
import ast
import pandas as pd
import os
import numpy as np

### Load Data

In [251]:
#set szenario
szenario_year = '2035'
szenario_name = 'run_' + szenario_year + '_AP'

In [252]:
#output specifications
output_file_path = os.path.join('..', '..', '01_data', '02_output_data', '02_unidirectional_results', '02_paper_ESR', '02_prepared_results')
outout_file_name = '\flow_analysis_ESR_2026_' + szenario_name + '.xlsx'
#create full ouput paths
output_file_path_excel  = output_file_path + outout_file_name
full_output_file_path = os.path.abspath(os.path.join(os.getcwd(), output_file_path_excel))

In [253]:
# Specify the path to the results Excel file
input_results_file_path_results = os.path.join('..', '..','01_data', '02_output_data', '02_unidirectional_results', '02_paper_ESR', '01_raw_results')
results_file_name = '\outputs_ESR_2026_' + szenario_name + '.xlsx'


# Specify the path to the input data Excel file
input_file_path = os.path.join('..', '..','01_data', '01_input_data', '02_processed', '02_paper_ESR')
input_file_name = '\inputs_ESR_2026_' + szenario_name + '.xlsx'

# Specify the path to the raw edges data Excel file
file_name_edges_cap_cost_raw = '\inputs_ESR_2026_edges_raw.xlsx'

#create full input paths
#results of the optimization
input_results_file_path_excel  = input_results_file_path_results + results_file_name
full_input_results_path = os.path.abspath(os.path.join(os.getcwd(), input_results_file_path_excel))

#inputs of the optimization
input_file_path_excel  = input_file_path + input_file_name
full_input_file_path = os.path.abspath(os.path.join(os.getcwd(), input_file_path_excel))

#edges raw path
file_path_edges_cap_costs_raw  = input_file_path + file_name_edges_cap_cost_raw
full_file_path_edges_cap_costs_raw = os.path.abspath(os.path.join(os.getcwd(), file_path_edges_cap_costs_raw))

In [254]:
results_methan_flow_raw_df = pd.read_excel(full_input_results_path, sheet_name='flows_methane_edges')

### code

In [255]:
def filter_flows_between_regions(
    df: pd.DataFrame,
    from_regions,
    to_regions,
    edge_col: str = "Edge",
    flow_col: str = "Flow",
    commodity: str | None = None,
    drop_zero_flows: bool = False,
):
    """
    Return all rows where Edge goes from any region in `from_regions`
    to any region in `to_regions`.

    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe.
    from_regions : str or list-like
        Origin region(s), e.g. "UA" or ["UA", "MD"].
    to_regions : str or list-like
        Destination region(s), e.g. ["CZ", "BE", "DE"].
    edge_col : str
        Column containing edges, e.g. ('UA', 'SK') or "('UA', 'SK')".
    flow_col : str
        Flow column name.
    commodity : str or None
        Optional filter, e.g. "Methane".
    drop_zero_flows : bool
        If True, remove rows where Flow == 0.

    Returns
    -------
    pd.DataFrame
        Filtered dataframe with extra columns: origin, destination.
    """
    # Normalize inputs
    if isinstance(from_regions, str):
        from_regions = [from_regions]
    if isinstance(to_regions, str):
        to_regions = [to_regions]

    from_regions = set(from_regions)
    to_regions = set(to_regions)

    out = df.copy()

    # Parse edge values safely whether they are tuples or strings
    def parse_edge(x):
        if isinstance(x, tuple) and len(x) == 2:
            return x
        if isinstance(x, str):
            return ast.literal_eval(x)
        raise ValueError(f"Unsupported edge format: {x!r}")

    out[["origin", "destination"]] = out[edge_col].apply(
        lambda x: pd.Series(parse_edge(x))
    )

    mask = out["origin"].isin(from_regions) & out["destination"].isin(to_regions)

    if commodity is not None:
        mask &= out["Commodity"].eq(commodity)

    if drop_zero_flows:
        mask &= out[flow_col].ne(0)

    return out.loc[mask].reset_index(drop=True)



def _parse_edge(x):
    """Parse Edge values that are either tuples or strings like "('DE', 'CH')"."""
    if isinstance(x, tuple) and len(x) == 2:
        return x
    if isinstance(x, str):
        return ast.literal_eval(x)
    raise ValueError(f"Unsupported edge format: {x!r}")


def edges_above_share(
    df: pd.DataFrame,
    regions,
    min_share: float,
    edge_col: str = "Edge",
    share_col: str = "Share",
    commodity: str | None = None,
    connection_mode: str = "both",
    drop_zero_flows: bool = True,
):
    """
    Filter edges by region connectivity and keep only rows with Share >= min_share.

    connection_mode:
        'both'   -> both origin and destination in regions
        'import' -> origin outside, destination inside
        'export' -> origin inside, destination outside
        'either' -> at least one side inside
    """
    regions = set(regions)
    out = df.copy()

    out[["origin", "destination"]] = out[edge_col].apply(
        lambda x: pd.Series(_parse_edge(x))
    )

    origin_in = out["origin"].isin(regions)
    destination_in = out["destination"].isin(regions)

    if connection_mode == "both":
        mask = origin_in & destination_in
    elif connection_mode == "import":
        mask = (~origin_in) & destination_in
    elif connection_mode == "export":
        mask = origin_in & (~destination_in)
    elif connection_mode == "either":
        mask = origin_in | destination_in
    else:
        raise ValueError("connection_mode must be: 'both', 'import', 'export', or 'either'")

    if commodity is not None:
        mask &= out["Commodity"].eq(commodity)

    if drop_zero_flows:
        mask &= out["Flow"].ne(0)

    out = out.loc[mask].copy()

    if share_col not in out.columns:
        raise KeyError(f"'{share_col}' not found in dataframe.")

    out = out.loc[out[share_col] >= min_share].sort_values(
        by=share_col, ascending=False
    )

    return out.reset_index(drop=True)

def lng_import_supplier_shares(
    df: pd.DataFrame,
    lng_import_nodes,
    edge_col: str = "Edge",
    flow_col: str = "Flow",
    share_col: str = "Share",
    commodity: str | None = None,
    min_edge_share: float | None = None,
    drop_zero_flows: bool = True,
):
    """
    Return LNG import edges plus supplier shares into each LNG import node.

    Output columns include:
        origin, destination, Flow, Share, supplier_flow, node_total_flow, supplier_share

    supplier_share = supplier_flow / total_flow_into_that_LNG_node
    """
    lng_import_nodes = set(lng_import_nodes)
    out = df.copy()

    out[["origin", "destination"]] = out[edge_col].apply(
        lambda x: pd.Series(_parse_edge(x))
    )

    mask = out["destination"].isin(lng_import_nodes)

    if commodity is not None:
        mask &= out["Commodity"].eq(commodity)

    if drop_zero_flows:
        mask &= out[flow_col].ne(0)

    if min_edge_share is not None:
        if share_col not in out.columns:
            raise KeyError(f"'{share_col}' not found in dataframe.")
        mask &= out[share_col].ge(min_edge_share)

    out = out.loc[mask].copy()

    # Aggregate supplier flow by (LNG node, origin)
    supplier_totals = (
        out.groupby(["destination", "origin"], as_index=False)[flow_col]
        .sum()
        .rename(columns={flow_col: "supplier_flow"})
    )

    # Total inflow into each LNG node
    node_totals = (
        supplier_totals.groupby("destination", as_index=False)["supplier_flow"]
        .sum()
        .rename(columns={"supplier_flow": "node_total_flow"})
    )

    supplier_totals["supplier_share"] = (
        supplier_totals["supplier_flow"]
        / supplier_totals.groupby("destination")["supplier_flow"].transform("sum")
    )

    # Merge supplier shares back onto the edge-level rows
    out = out.merge(
        supplier_totals,
        on=["destination", "origin"],
        how="left",
    ).merge(
        node_totals,
        on="destination",
        how="left",
    )

    out = out.sort_values(
        ["destination", "supplier_share", flow_col],
        ascending=[True, False, False],
    )

    return out.reset_index(drop=True)

### execution

In [256]:
#regional groups
europe = [
    "CZ",
    "BE",
    "DE",
    "FR",
    "NL",
    "AT",
    "SI",
    "HR",
    "PL",
    "SK",
    "IT",
    "ES",
    "RO",
    "HU",
    "RS",
    "BA",
    "MD",
    "UA",
    "BG",
    "CH",
    "UK",
    "LV",
    "TR",
    "FI",
]

northern_africa = [
    "DZ",
    "TN",
    "LY",
    "MA",
]

lng_import_nodes = [
    "EL_LNG_imp",
    "IT_LNG_imp",
    "BE_LNG_imp",
    "TR_LNG_imp",
    "ES_LNG_imp",
    "HR_LNG_imp",
    "PL_LNG_imp",
    "UK_LNG_imp",
    "LT_LNG_imp",
    "NL_LNG_imp",
    "FR_LNG_imp",
    "PT_LNG_imp",
    "DE_LNG_imp",
    "FI_LNG_imp",
    "IE_LNG_imp",
    "LV_LNG_imp",
    "EE_LNG_imp",
]

lng_import_countries = [
    "EL",
    "IT",
    "BE",
    "TR",
    "ES",
    "HR",
    "PL",
    "UK",
    "LT",
    "NL",
    "FR",
    "PT",
    "DE",
    "FI",
    "IE",
    "LV",
    "EE",
]

In [257]:
#Russia to Europe
ru_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions="RU",
    to_regions=europe,
    commodity="Methane"
)

#Northern Africa to Europe
NA_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions=northern_africa,
    to_regions=europe,
    commodity="Methane"
)

#Norway to Europe
no_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions="NO",
    to_regions=europe,
    commodity="Methane"
)

#Caspean Region to Europe
cr_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions="CR",
    to_regions=europe,
    commodity="Methane"
)

In [258]:
ru_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('RU', 'FI')",0.0,0.0,RU,FI
1,Methane,"('RU', 'LV')",0.0,0.0,RU,LV
2,Methane,"('RU', 'TR')",0.0,0.0,RU,TR
3,Methane,"('RU', 'UA')",0.0,0.0,RU,UA
4,Methane,"('RU', 'DE')",0.0,0.0,RU,DE


In [259]:
total_RU_to_europe = ru_to_europe["Flow"].sum()
total_RU_to_europe

np.float64(0.0)

In [260]:
RU_real = 167*9.77*1000
reduction = (1-(total_RU_to_europe/RU_real))*100
round(reduction, 2)

np.float64(100.0)

In [261]:
NA_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('DZ', 'ES')",123041.500000,1.000000,DZ,ES
1,Methane,"('MA', 'ES')",9671.536315,0.830118,MA,ES
2,Methane,"('TN', 'IT')",378127.507427,0.897717,TN,IT
3,Methane,"('LY', 'IT')",60977.253841,0.338386,LY,IT


In [262]:
total_NA_to_europe = NA_to_europe["Flow"].sum()
total_NA_to_europe

np.float64(571817.7975819347)

In [263]:
no_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('NO', 'DE')",402713.436028,0.899938,NO,DE
1,Methane,"('NO', 'FR')",0.000000,0.000000,NO,FR
2,Methane,"('NO', 'NL')",107495.158756,0.305632,NO,NL
3,Methane,"('NO', 'UK')",241360.556840,0.470314,NO,UK
4,Methane,"('NO', 'BE')",0.000000,0.000000,NO,BE


In [264]:
total_no_to_europe = no_to_europe["Flow"].sum()
total_no_to_europe

np.float64(751569.1516242205)

In [265]:
cr_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('CR', 'TR')",293100.0,1.0,CR,TR


In [266]:
total_cr_to_europe = cr_to_europe["Flow"].sum()
total_cr_to_europe

np.float64(293099.9999999999)

In [267]:
lng_to_europe = filter_flows_between_regions(
    results_methan_flow_raw_df,
    from_regions=lng_import_nodes,
    to_regions=lng_import_countries,
    commodity="Methane",
    drop_zero_flows=True
)

In [268]:
lng_to_europe

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('ES_LNG_imp', 'ES')",16894.098217,0.025770,ES_LNG_imp,ES
1,Methane,"('EE_LNG_imp', 'EE')",1889.421018,0.077356,EE_LNG_imp,EE
2,Methane,"('IE_LNG_imp', 'IE')",21985.591341,0.865506,IE_LNG_imp,IE
3,Methane,"('PT_LNG_imp', 'PT')",22886.974711,0.308234,PT_LNG_imp,PT
4,Methane,"('HR_LNG_imp', 'HR')",19173.951093,0.377410,HR_LNG_imp,HR
5,Methane,"('TR_LNG_imp', 'TR')",51338.277261,0.102591,TR_LNG_imp,TR
6,Methane,"('LV_LNG_imp', 'LV')",3918.517715,0.267384,LV_LNG_imp,LV
7,Methane,"('FI_LNG_imp', 'FI')",6101.247037,0.138775,FI_LNG_imp,FI
8,Methane,"('LT_LNG_imp', 'LT')",7996.909983,0.204629,LT_LNG_imp,LT
9,Methane,"('EL_LNG_imp', 'EL')",79536.331764,0.361495,EL_LNG_imp,EL


In [269]:
total_lng_to_europe = lng_to_europe["Flow"].sum()
total_lng_to_europe

np.float64(577594.50190418)

In [270]:
europe_pipeline_use = edges_above_share(
    results_methan_flow_raw_df,
    regions=europe,
    min_share=0.8,
    commodity="Methane",
    connection_mode="either"
)

In [271]:
europe_pipeline_use

,Commodity,Edge,Flow,Share,origin,destination
0,Methane,"('DZ', 'ES')",123041.500000,1.000000,DZ,ES
1,Methane,"('IT', 'AT')",70828.250000,1.000000,IT,AT
2,Methane,"('IT', 'SI')",14231.350000,1.000000,IT,SI
3,Methane,"('CR', 'TR')",293100.000000,1.000000,CR,TR
4,Methane,"('CH', 'FR')",36500.000000,1.000000,CH,FR
5,Methane,"('HR', 'HU')",18578.500000,1.000000,HR,HU
6,Methane,"('PL_LNG_imp', 'PL')",133548.409814,0.949252,PL_LNG_imp,PL
7,Methane,"('NO', 'DE')",402713.436028,0.899938,NO,DE
8,Methane,"('TN', 'IT')",378127.507427,0.897717,TN,IT
9,Methane,"('MA', 'ES')",9671.536315,0.830118,MA,ES


In [272]:
lng_shares = lng_import_supplier_shares(
    results_methan_flow_raw_df,
    lng_import_nodes=lng_import_nodes,
    commodity="Methane",
    min_edge_share=0.0
)

In [249]:
lng_shares

,Commodity,Edge,Flow,Share,origin,destination,supplier_flow,supplier_share,node_total_flow
0,Methane,"('USA_LNG_exp', 'BE_LNG_imp')",93427.662759,0.000093,USA_LNG_exp,BE_LNG_imp,93427.662759,1.000000,93427.662759
1,Methane,"('USA_LNG_exp', 'DE_LNG_imp')",242697.281208,0.000243,USA_LNG_exp,DE_LNG_imp,242697.281208,1.000000,242697.281208
2,Methane,"('USA_LNG_exp', 'EE_LNG_imp')",2907.671867,0.000003,USA_LNG_exp,EE_LNG_imp,2907.671867,1.000000,2907.671867
3,Methane,"('AF_LNG_exp', 'EL_LNG_imp')",105281.932023,0.000105,AF_LNG_exp,EL_LNG_imp,105281.932023,1.000000,105281.932023
4,Methane,"('AF_LNG_exp', 'ES_LNG_imp')",57639.493627,0.000058,AF_LNG_exp,ES_LNG_imp,57639.493627,0.582845,98893.380319
5,Methane,"('USA_LNG_exp', 'ES_LNG_imp')",41253.886692,0.000041,USA_LNG_exp,ES_LNG_imp,41253.886692,0.417155,98893.380319
6,Methane,"('USA_LNG_exp', 'FI_LNG_imp')",9389.344243,0.000009,USA_LNG_exp,FI_LNG_imp,9389.344243,1.000000,9389.344243
7,Methane,"('USA_LNG_exp', 'FR_LNG_imp')",292913.151331,0.000293,USA_LNG_exp,FR_LNG_imp,292913.151331,1.000000,292913.151331
8,Methane,"('AF_LNG_exp', 'HR_LNG_imp')",35077.658398,0.000035,AF_LNG_exp,HR_LNG_imp,35077.658398,1.000000,35077.658398
9,Methane,"('USA_LNG_exp', 'IE_LNG_imp')",25402.000000,0.000025,USA_LNG_exp,IE_LNG_imp,25402.000000,1.000000,25402.000000


In [250]:
fr_lng = lng_shares[lng_shares["destination"] == "FR_LNG_imp"]

In [69]:
fr_lng

,Commodity,Edge,Flow,Share,origin,destination,supplier_flow,supplier_share,node_total_flow
2,Methane,"('USA_LNG_exp', 'FR_LNG_imp')",122557.651478,0.000123,USA_LNG_exp,FR_LNG_imp,122557.651478,1.0,122557.651478
